# Cox Proportional Hazards Analysis for OSS Project Survival

This notebook demonstrates the Cox survival analysis to test whether knowledge redundancy has an inverted-U relationship with OSS project survival after founder departure.

**Research Question**: Does knowledge redundancy in open-source projects follow an inverted-U pattern with project survival after founder departure?

**Method**: Cox proportional hazards models with linear and quadratic terms for knowledge redundancy, controlling for project characteristics (stars, commits, contributors, language).

In [ ]:
import subprocess, sys

def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# lifelines - survival analysis library (NOT pre-installed on Colab)
_pip('lifelines==0.30.0')
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test, logrank_test
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import os
import resource

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print("All imports successful!")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-a68c06-knowledge-redundancy-predicts-oss/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined.")

In [ ]:
DATA = load_data()
print(f"Loaded data with {len(DATA['datasets'][0]['examples'])} examples")
print(f"Dataset: {DATA['datasets'][0]['dataset']}")

## Configuration

Minimum parameters for demo - just load data and run analysis on the mini dataset.

In [ ]:
RAM_BUDGET = 2 * 1024**3  # 2GB for demo
resource.setrlimit(resource.RLIMIT_AS, (RAM_BUDGET * 3, RAM_BUDGET * 3))

print(f"Memory limit set to {RAM_BUDGET // 1024**3}GB")

## 1. Load and Parse Data

Parse the input JSON strings and create records with knowledge redundancy scores, project characteristics, and output (survived/died).

In [ ]:
logger.info("Loading and parsing demo data")

examples = DATA['datasets'][0]['examples']
logger.info(f"Loaded {len(examples)} examples from JSON")

records = []
for i, ex in enumerate(examples):
    try:
        input_dict = json.loads(ex['input'])
        record = {
            'knowledge_redundancy_score': input_dict['knowledge_redundancy_score'],
            'stars': input_dict['stars'],
            'language_encoded': input_dict.get('language_encoded', 1),
            'total_commits': input_dict['total_commits'],
            'top_contributors_count': input_dict['top_contributors_count'],
            'pre_departure_commits_per_month': input_dict['pre_departure_commits_per_month'],
            'post_departure_commits_per_month': input_dict.get('post_departure_commits_per_month', 0),
            'output': ex['output'],
            'metadata_has_departure': ex.get('metadata_has_departure', True),
            'metadata_repo_id': ex.get('metadata_repo_id', f'repo-{i}'),
            'metadata_language': ex.get('metadata_language', 'unknown')
        }
        records.append(record)
    except Exception as e:
        logger.error(f"Failed to parse example {i}: {e}")
        continue

df = pd.DataFrame(records)
logger.info(f"Parsed {len(df)} valid records")
logger.info(f"Output distribution: {df['output'].value_counts().to_dict()}")
logger.info(f"Has departure distribution: {df['metadata_has_departure'].value_counts().to_dict()}")

df

## 2. Prepare Survival Data

Create survival analysis variables:
- T = duration (time until event or censoring)
- E = event indicator (1 if project died, 0 if survived/censored)
- KR = knowledge redundancy score
- KR_squared = quadratic term (centered)
- Control variables: log-transformed stars and commits, contributor count, language dummies

In [ ]:
logger.info("Preparing survival analysis variables")

df_departed = df[df['metadata_has_departure'] == True].copy()
logger.info(f"Repos with founder departure: {len(df_departed)}")

df_departed['T'] = 12.0
df_departed['E'] = 0

for idx in df_departed.index:
    if df_departed.loc[idx, 'output'] == 'died':
        pre = df_departed.loc[idx, 'pre_departure_commits_per_month']
        post = df_departed.loc[idx, 'post_departure_commits_per_month']
        if pre > 0 and post < 0.1 * pre:
            df_departed.loc[idx, 'T'] = 4.0
        else:
            df_departed.loc[idx, 'T'] = 6.0
        df_departed.loc[idx, 'E'] = 1

logger.info(f"Died cases (E=1): {(df_departed['E'] == 1).sum()}")
logger.info(f"Survived cases (E=0): {(df_departed['E'] == 0).sum()}")

kr_mean = df_departed['knowledge_redundancy_score'].mean()
df_departed['KR'] = df_departed['knowledge_redundancy_score']
df_departed['KR_centered'] = df_departed['KR'] - kr_mean
df_departed['KR_squared'] = df_departed['KR_centered'] ** 2

logger.info(f"KR mean for centering: {kr_mean:.4f}")

df_departed['stars_log'] = np.log(df_departed['stars'] + 1)
df_departed['total_commits_log'] = np.log(df_departed['total_commits'] + 1)

df_departed['language_str'] = df_departed['language_encoded'].astype(str)
language_dummies = pd.get_dummies(df_departed['language_str'], prefix='lang')
df_departed = pd.concat([df_departed, language_dummies], axis=1)

df_survival = df_departed

logger.info(f"Survival data prepared: {len(df_survival)} samples")
logger.info(f"  - KR range: [{df_departed['KR'].min():.3f}, {df_departed['KR'].max():.3f}]")
logger.info(f"  - Events (died): {(df_departed['E'] == 1).sum()}")
logger.info(f"  - Censored (survived): {(df_departed['E'] == 0).sum()}")

df_survival[['KR', 'T', 'E', 'stars_log', 'total_commits_log', 'top_contributors_count']].head()

## 3. Fit Cox Proportional Hazards Models

Fit two models:
1. **Linear model** (baseline): hazard = baseline * exp(β1*KR + β_controls*controls)
2. **Quadratic model** (tests inverted-U): hazard = baseline * exp(β1*KR + β2*KR^2 + β_controls*controls)

Compare models using likelihood ratio test.

In [ ]:
logger.info("Fitting Cox proportional hazards models")

base_cols = ['T', 'E', 'KR_centered', 'KR_squared', 'stars_log',
            'total_commits_log', 'top_contributors_count',
            'pre_departure_commits_per_month']

lang_cols = [col for col in df_survival.columns if col.startswith('lang_')]
all_cols = base_cols + lang_cols

df_model = df_survival[all_cols].copy()
df_model = df_model.dropna()
logger.info(f"Model data after removing NA: {len(df_model)} samples")

n_events = (df_model['E'] == 1).sum()
logger.info(f"Number of events (deaths): {n_events}")

logger.info("Fitting Model 1: Linear-only Cox model (baseline)")
cph_linear = CoxPHFitter(penalizer=0.01)

linear_formula = 'KR_centered + stars_log + total_commits_log + '
linear_formula += 'top_contributors_count + pre_departure_commits_per_month + '
linear_formula += ' + '.join([f'C({col})' for col in lang_cols])

cph_linear.fit(df_model, duration_col='T', event_col='E', formula=linear_formula)
logger.info("Model 1 (Linear) fitted successfully")
logger.info(f"Linear model concordance: {cph_linear.concordance_index_:.4f}")

logger.info("Fitting Model 2: Quadratic Cox model (tests inverted-U)")
cph_quadratic = CoxPHFitter(penalizer=0.01)

quad_formula = 'KR_centered + KR_squared + stars_log + total_commits_log + '
quad_formula += 'top_contributors_count + pre_departure_commits_per_month + '
quad_formula += ' + '.join([f'C({col})' for col in lang_cols])

cph_quadratic.fit(df_model, duration_col='T', event_col='E', formula=quad_formula)
logger.info("Model 2 (Quadratic) fitted successfully")
logger.info(f"Quadratic model concordance: {cph_quadratic.concordance_index_:.4f}")

lr_test_stat = 2 * (cph_quadratic.log_likelihood_ - cph_linear.log_likelihood_)
lr_p_value = 1 - stats.chi2.cdf(lr_test_stat, df=1)

model_comparison = {
    'LR_test_statistic': lr_test_stat,
    'LR_test_p_value': lr_p_value,
    'AIC_linear': cph_linear.AIC_partial_,
    'AIC_quadratic': cph_quadratic.AIC_partial_
}

logger.info(f"Likelihood ratio test: statistic={lr_test_stat:.4f}, p={lr_p_value:.4f}")
print("\nModel fitted successfully!")
print(f"Linear model concordance: {cph_linear.concordance_index_:.4f}")
print(f"Quadratic model concordance: {cph_quadratic.concordance_index_:.4f}")
print(f"LR test p-value: {lr_p_value:.4f}")

## 4. Test Inverted-U Hypothesis

Test whether knowledge redundancy has an inverted-U relationship with survival:
- H0: β2 = 0 (no quadratic relationship)
- H1: β2 > 0 (positive quadratic term, indicating U-shaped hazard = inverted-U survival)

Criteria for confirmation:
1. β2 > 0 (positive quadratic coefficient)
2. p-value < 0.05 (statistically significant)
3. Turning point in [0, 1] range

In [ ]:
logger.info("Testing inverted-U hypothesis")

coef = cph_quadratic.params_
beta1 = coef['KR_centered']
beta2 = coef['KR_squared']

logger.info(f"Coefficient β1 (linear KR): {beta1:.4f}")
logger.info(f"Coefficient β2 (quadratic KR^2): {beta2:.4f}")

p_value = cph_quadratic.summary.loc['KR_squared', 'p']
logger.info(f"β2 p-value: {p_value:.4f}")

if beta2 != 0:
    turning_point = -beta1 / (2 * beta2)
else:
    turning_point = np.nan

logger.info(f"Turning point (KR for max hazard): {turning_point:.4f}")

turning_point_in_range = 0 <= turning_point <= 1 if not np.isnan(turning_point) else False
inverted_U_confirmed = (beta2 > 0) and (p_value < 0.05) and turning_point_in_range

kr_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
hazard_ratios = {}

for kr in kr_values:
    kr_centered = kr - df_survival['KR'].mean()
    log_hr = beta1 * kr_centered + beta2 * (kr_centered ** 2)
    hr = np.exp(log_hr)
    hazard_ratios[f'at_KR_{kr}'] = float(hr)

logger.info(f"Hazard ratios at key KR values: {hazard_ratios}")
logger.info(f"Inverted-U hypothesis confirmed: {inverted_U_confirmed}")

print("\n=== HYPOTHESIS TEST RESULTS ===")
print(f"β1 (linear KR): {beta1:.4f}")
print(f"β2 (quadratic KR^2): {beta2:.4f}")
print(f"β2 p-value: {p_value:.4f}")
print(f"Turning point: {turning_point:.4f}")
print(f"Inverted-U confirmed: {inverted_U_confirmed}")

## 5. Results Visualization

Generate diagnostic plots to visualize the survival analysis results.

In [ ]:
logger.info("Generating diagnostic plots")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

fig, ax = plt.subplots()

kr_mean = df_survival['KR'].mean()
colors = ['red', 'green', 'blue', 'orange']

for i, kr in enumerate([0.2, 0.4, 0.6, 0.8]):
    sample_df = df_model.iloc[[0]].copy()
    sample_df['KR_centered'] = kr - kr_mean
    sample_df['KR_squared'] = (kr - kr_mean) ** 2
    
    try:
        surv_func = cph_quadratic.predict_survival_function(sample_df)
        ax.plot(surv_func.index, surv_func.values.flatten(),
               label=f'KR={kr}', color=colors[i], linewidth=2)
    except Exception as e:
        logger.error(f"Failed to plot survival curve for KR={kr}: {e}")

ax.set_xlabel('Time (months)', fontsize=12)
ax.set_ylabel('Survival Probability', fontsize=12)
ax.set_title('Survival Curves by Knowledge Redundancy Level', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()

kr_range = np.linspace(0, 1, 100)
hr_values = []

for kr in kr_range:
    kr_c = kr - kr_mean
    log_hr = beta1 * kr_c + beta2 * kr_c**2
    hr_values.append(np.exp(log_hr))

ax.plot(kr_range, hr_values, linewidth=2, color='blue')
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='HR=1')
if not np.isnan(turning_point):
    ax.axvline(x=turning_point, color='green', linestyle='--', alpha=0.5,
              label=f"Turning point={turning_point:.2f}")
ax.set_xlabel('Knowledge Redundancy Score', fontsize=12)
ax.set_ylabel('Hazard Ratio (vs reference)', fontsize=12)
ax.set_title('Hazard Ratio vs Knowledge Redundancy', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Plots generated successfully!")

## 6. Summary of Results

Print a comprehensive summary of the analysis results.

In [ ]:
print("=" * 70)
print("COX PROPORTIONAL HAZARDS SURVIVAL ANALYSIS SUMMARY")
print("=" * 70)

print("\n1. DATA SUMMARY:")
print(f"   Total repos: {len(df)}")
print(f"   Repos with founder departure: {len(df_survival)}")
print(f"   Died (events): {(df_survival["E"] == 1).sum()}")
print(f"   Survived (censored): {(df_survival["E"] == 0).sum()}")
print(f"   KR mean: {df_survival["KR"].mean():.4f}")
print(f"   KR std: {df_survival["KR"].std():.4f}")

print("\n2. QUADRATIC MODEL RESULTS:")
print(f"   Beta1 (linear KR): {beta1:.4f}")
print(f"   Beta2 (quadratic KR^2): {beta2:.4f}")
print(f"   Beta2 p-value: {p_value:.4f}")
print(f"   Turning point (KR for max hazard): {turning_point:.4f}")

print("\n3. HYPOTHESIS TEST (Inverted-U):")
print(f"   Inverted-U confirmed: {inverted_U_confirmed}")
print(f"   Criteria: β2 > 0, p < 0.05, turning point in [0,1]")

print("\n4. MODEL COMPARISON:")
print(f"   Linear model concordance: {cph_linear.concordance_index_:.4f}")
print(f"   Quadratic model concordance: {cph_quadratic.concordance_index_:.4f}")
print(f"   Partial AIC: Linear={cph_linear.AIC_partial_:.2f}, Quadratic={cph_quadratic.AIC_partial_:.2f}")
print(f"   LR test p-value: {model_comparison["LR_test_p_value"]:.4f}")

print("\n5. HAZARD RATIOS AT KEY KR VALUES:")
for kr, hr in hazard_ratios.items():
    print(f"   {kr}: HR = {hr:.4f}")

print("=" * 70)